In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# This script loads a custom 96w deep well plate 2.2mL v-bottom and 
# Tests the dimensions of the plate so I don't have to repeatedly run lh.setup

# ── imports ──────────────────────────────────────────────
from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    MFX_CAR_L5_base,
    TIP_CAR_480_A00
)
from pylabrobot.resources.hamilton.mfx_modules import (
    Hamilton_MFX_plateholder_DWP_metal_tapped
)
from pylabrobot.resources import (
    TIP_50ul_w_filter,
                 HTF             # 50 µL filter tip rack
)
import asyncio

# ── build LH + deck ──────────────────────────────────────
backend = STARBackend()
lh      = LiquidHandler(backend=backend, deck=STARLetDeck())

# ── initialise hardware (Autoload) ───────────────────────
await lh.setup(skip_autoload=True)

# ── tip carrier with 50 µL tips ───────────────────────────
tip_car = TIP_CAR_480_A00("tip_car")
tip_car[0] = HTF(name="tips_00")
tip_car[1] = TIP_50ul_w_filter(name="tips_01")
lh.deck.assign_child_resource(tip_car, rails=25)


# ── module → carrier → deck ──────────────────────────────
dwp_mod   = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_1")
other_mod = Hamilton_MFX_plateholder_DWP_metal_tapped("dwp_mod_2")
flex_car  = MFX_CAR_L5_base("flex_car_1", modules={0: dwp_mod, 1: other_mod})
lh.deck.assign_child_resource(flex_car, rails=13)

In [3]:
# ── pick up a single 50 µL tip on channel 3 ─────────────
tiprack = lh.deck.get_resource("tips_01")
longrack = lh.deck.get_resource("tips_00")


In [4]:
from typing import Optional

from pylabrobot.resources.height_volume_functions import (
  compute_height_from_volume_rectangle,
  compute_volume_from_height_rectangle,
)
from pylabrobot.resources.plate import Lid, Plate
from pylabrobot.resources.utils import create_ordered_items_2d
from pylabrobot.resources.well import (
  CrossSectionType,
  Well,
  WellBottomType,
)

from pylabrobot.resources.bioer import BioER_96_wellplate_Vb_2200ul

def BioER_96_wellplate_Vb_2200ul(name: str) -> Plate:
  """BioER Cat. No. BSH06M1T-A (KingFisher-compatible)
  Spec: https://en.bioer.com/uploadfiles/2024/05/20240513165756879.pdf
  """
  return Plate(
    name=name,
    size_x=127.1,  # from spec
    size_y=85.0,  # from spec
    size_z=44.2,  # from spec
    lid=None,
    model=BioER_96_wellplate_Vb_2200ul.__name__,
    ordered_items=create_ordered_items_2d(
      Well,
      size_x=8.25,  # from spec (inner well width)
      size_y=8.25,  # from spec (inner well length)
      size_z=42.4,  # measured (well depth)
      dx=9.5,  # measured (column pitch)
      dy=7.5,  # measured (row pitch)
      dz=6,  # measured (expected to be 44.2-42.4-0.8=1, but 6 optimal on Hamilton_MFX_plateholder_DWP_metal_tapped )
      material_z_thickness=0.8,  # measured
      item_dx=9.0,  # measured 
      item_dy=9.0,  # measured 
      num_items_x=12,  # from spec
      num_items_y=8,  # from spec
      cross_section_type=CrossSectionType.RECTANGLE,
      bottom_type=WellBottomType.V,
      max_volume=2200,  # from spec (2.2 mL)
    ),
  )


In [5]:
# ── add the 2.2 mL deep well plate inside the DWP module ──
dp1 = BioER_96_wellplate_Vb_2200ul(name="dp1")
dwp_mod.assign_child_resource(dp1)

In [20]:
st = 3
# ht = 5

AT = list(range(8))

# await lh.pick_up_tips(tiprack["A1"], use_channels=[4])
# await lh.pick_up_tips(tiprack["A6:H6"], use_channels=AT)
# await lh.pick_up_tips(longrack["A1:H1"], use_channels=AT)
await lh.pick_up_tips(longrack["A1"], use_channels=[4])

await lh.aspirate(dp1["A1"], vols=[0], use_channels=[4], liquid_height=[1], settling_time=[st])
await lh.aspirate(dp1["B1"], vols=[0], use_channels=[4], liquid_height=[1], settling_time=[st])
await lh.aspirate(dp1["C1"], vols=[0], use_channels=[4], liquid_height=[1], settling_time=[st])
await lh.aspirate(dp1["D1"], vols=[0], use_channels=[4], liquid_height=[1], settling_time=[st])
# await lh.aspirate(dp1["A1:H1"], vols=[0]*8, use_channels=AT, liquid_height=[0]*8, settling_time=[st]*8)
# await lh.aspirate(dp1["A12"], vols=[0], use_channels=[4], liquid_height=[0], settling_time=[st])
# await lh.aspirate(dp1["A12:H12"], vols=[0]*8, use_channels=AT, liquid_height=[0]*8, settling_time=[st]*8)



In [27]:
import time
a1 = dp1["A1"][0]
coord = a1.get_absolute_location(x="c", y="c", z="cavity_bottom")
print(f"Absolute coordinates of A1 cavity bottom: {coord}")
CHANNEL = 4

# move in X, Y, then Z
await lh.move_channel_x(CHANNEL, coord.x)
await lh.move_channel_y(CHANNEL, coord.y)
await lh.move_channel_z(CHANNEL, coord.z)
time.sleep(2)
# await lh.move_channel_z(CHANNEL, 270) # for 50ul filtered
await lh.move_channel_z(CHANNEL, 240) # for 1000ul filtered
time.sleep(2)
await lh.aspirate(dp1["A1"], vols=[0], use_channels=[4], liquid_height=[0], settling_time=[2])

Absolute coordinates of A1 cavity bottom: Coordinate(387.625, 146.625, 186.010)


In [7]:
# loc_top = deep_plate7["H1"][0].get_absolute_location(z="top")

# await lh.prepare_for_manual_channel_operation(3)

# # move in X, Y, Z
# await lh.move_channel_x(3, loc_top.x)
# await lh.move_channel_y(3, loc_top.y)
# await lh.move_channel_z(3, loc_top.z)

# loc_bot = dp10["H1"][0].get_absolute_location(z="bottom")

# # move in X, Y, Z
# await lh.move_channel_x(3, loc_bot.x)
# await lh.move_channel_y(3, loc_bot.y)
# await lh.move_channel_z(3, loc_bot.z)
# 
# print (loc_bot.x, loc_bot.y, loc_bot.z)

In [28]:
# await lh.drop_tips(tiprack["A1"], use_channels=[4])
await lh.drop_tips(longrack["A1"], use_channels=[4])
# await lh.drop_tips(tiprack["A2:H2"], use_channels=AT)
# await lh.discard_tips()
# await lh.stop()